# Exploratory Data Analysis

This notebook focuses on dataset quality, class balance, transaction behavior,
and leakage-safe temporal feature inspection.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from fraudshield.config.settings import get_settings  # noqa: E402
from fraudshield.feature_engineering.transaction_features import (  # noqa: E402
    TransactionFeatureConfig,
    add_transaction_features,
)
from fraudshield.runtime.logging import configure_logging  # noqa: E402

settings = get_settings()
configure_logging(settings, component="notebook_eda")

sns.set_theme(style="whitegrid", context="talk")
data_path = PROJECT_ROOT / "data" / "raw" / "synthetic_fraud_data.csv"
df = pd.read_csv(data_path, parse_dates=["transaction_date"])
df["event_day"] = df["transaction_date"].dt.floor("D")
df.head()

In [ ]:
schema_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing_rate": df.isna().mean(),
        "n_unique": df.nunique(dropna=False),
    }
)
schema_summary

In [ ]:
target_summary = (
    df.groupby("fraud")
    .agg(
        transactions=("transaction_id", "count"),
        avg_amount=("amount", "mean"),
        median_amount=("amount", "median"),
        unique_users=("user_id", "nunique"),
        unique_merchants=("merchant_id", "nunique"),
    )
    .round(2)
)
target_summary

In [ ]:
daily_volume = df.groupby("event_day").agg(transactions=("transaction_id", "count"), fraud_rate=("fraud", "mean")).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
sns.lineplot(data=daily_volume, x="event_day", y="transactions", ax=axes[0], color="#1f77b4")
sns.lineplot(data=daily_volume, x="event_day", y="fraud_rate", ax=axes[1], color="#d62728")
axes[0].set_title("Daily Transaction Volume")
axes[1].set_title("Daily Fraud Rate")
axes[1].set_ylim(0, daily_volume["fraud_rate"].max() * 1.1)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(data=df, x="amount", hue="fraud", bins=40, stat="density", common_norm=False, ax=axes[0])
sns.boxplot(data=df, x="fraud", y="amount", ax=axes[1])
axes[0].set_title("Amount Distribution By Fraud Label")
axes[1].set_title("Amount Boxplot By Fraud Label")
plt.tight_layout()

In [ ]:
merchant_summary = (
    df.groupby("merchant_id")
    .agg(transactions=("transaction_id", "count"), fraud_rate=("fraud", "mean"))
    .query("transactions >= 20")
    .sort_values(["fraud_rate", "transactions"], ascending=[False, False])
    .head(10)
)
merchant_summary

In [ ]:
feature_df = add_transaction_features(
    df.copy(),
    TransactionFeatureConfig(windows=["1h", "24h", "7d"]),
)
engineered_columns = [
    "user_time_since_last_txn",
    "user_amount_zscore",
    "user_txn_count_1h",
    "merchant_fraud_rate_24h",
]
feature_df[["transaction_id", "transaction_date", *engineered_columns]].head(12)

## Analyst Notes

- Start with null rates and entity cardinality before fitting any model.
- Prefer time-aware features such as rolling counts and merchant fraud rates to
  prevent leakage from future observations.
- Keep a minimum support threshold when ranking merchants or users by fraud rate;
  otherwise the analysis overreacts to tiny sample sizes.